In [3]:
# === Core Python ===
import os
import glob
import collections
import cftime
from datetime import datetime
from typing import Tuple, Dict, Optional

# === Numerical & Data Handling ===
import numpy as np
import numpy.ma as ma
import pandas as pd
import xarray as xr
import xcdat as xcd
import xskillscore as xs
from xskillscore import rmse, pearson_r

# === Scientific Computation ===
from scipy.stats import pearsonr
from scipy.signal import butter, filtfilt, sosfilt, lfilter
from scipy.optimize import curve_fit
from skimage.feature import peak_local_max
import metpy.calc as mpcalc  # Meteorology/thermodynamics

# === Plotting & Visualization ===
import matplotlib.pyplot as plt
from matplotlib.pylab import rcParams
from matplotlib.patches import Polygon
import matplotlib.colors as mcolors
import matplotlib.path as mpath
import matplotlib.ticker as mticker
import matplotlib.gridspec as gridspec
import matplotlib.cm as cm
from matplotlib.lines import Line2D
from matplotlib.patches import Patch

# === Mapping & Cartography ===
import cartopy.crs as ccrs
import cartopy.feature as cfeature
import cartopy.mpl.ticker as cticker
from mpl_toolkits.axes_grid1.inset_locator import inset_axes

# === Visualization Helpers ===
import geocat.viz.util as gvutil
import geocat.viz as gv
import cmaps as gvcmaps  # Optional, check if available

from scipy.signal import butter, filtfilt
from sklearn.decomposition import PCA

# === Tropical Cyclone Analysis ===
from tropycal import tracks, utils

/qfs/people/zhan391/.conda/envs/e3sm_analysis/lib/python3.12/site-packages/esmpy/interface/loadESMF.py:94: VersionWarning: ESMF installation version 8.8.0, ESMPy version 8.8.0b0
  warnings.warn("ESMF installation version {}, ESMPy version {}".format(


In [4]:
class ObsDataReader:
    """
    Class to read and preprocess observational data with internal dataset registry.
    Supports regional mean extraction.
    """

    def __init__(self):
        self.obs_group = self._define_obs_group()

    def _define_obs_group(self):
        return {
            'ERA5': {
                'monthly': {
                    'path': '/compyfs/zhan391/acme_init/Observations/ERA5/monthly',
                    'template': 'ERA5_analysis_monthly_%(year).nc',
                    'period': '2001-2018'
                },
                '6hourly': {
                    'path': '/compyfs/zhan391/acme_init/Observations/ERA5/6hourly',
                    'template': 'ERA5.6hourly.en00.%(var).%(year)01-%(year)12.nc',
                    'period': '1979-2018'
                }
            },
            'GPCP': {
                'monthly': {
                    'path': '/compyfs/zhan391/acme_init/Observations/GPCP/monthly',
                    'template': 'PRECT.monthly.%(year).nc',
                    'period': '1979-2017'
                },
                'daily': {
                    'path': '/compyfs/zhan391/acme_init/Observations/GPCP/daily',
                    'template': 'PRECT.daily.%(year).nc',
                    'period': '2007-2011'
                }
            },
            'IMERG': {
                'daily': {
                    'path': '/compyfs/zhan391/acme_init/Observations/IMERG/daily',
                    'template': 'PRECT.daily.%(year).nc',
                    'period': '2007-2011'
                }
            }
        }

    @staticmethod
    def define_region(regnam='global'):
        reg_dict = {
            'global':     [(-90, 90), (-180, 180)],
            'NHMidLat':   [(25, 50), (-60, 150)],
            'Tropics':    [(-10, 10), (-90, 60)],
            'Atlantic':   [(5, 55), (-95, -40)],
            'CONUS':      [(25, 50), (-125, -95)],
            'Antarctic':  [(-90, -50), (-180, 180)],
            'PolarN':     [(50, 90), (-180, 180)],
            'Greenland':  [(60, 85), (-75, -10)],
        }
        if regnam not in reg_dict:
            raise ValueError(f"Region '{regnam}' not defined. Available: {list(reg_dict.keys())}")
        return reg_dict[regnam]

    def get_group(self, name=None):
        return self.obs_group.get(name) if name else self.obs_group

    def list_datasets(self):
        return list(self.obs_group.keys())

    def _ensure_datetime64(self, da: xr.DataArray) -> xr.DataArray:
        """Convert time to datetime64 if cftime is used."""
        if not isinstance(da.indexes['time'], pd.DatetimeIndex):
            da['time'] = da.indexes['time'].to_datetimeindex(time_unit='ns')
        return da

    def read(self, var, obsname, time, frequency=None, regnam='global', regional_mean=False):
        obs_group = self.obs_group.get(obsname)
        if obs_group is None:
            raise ValueError(f"Observational dataset '{obsname}' not found.")
        if frequency not in obs_group:
            raise ValueError(f"Frequency '{frequency}' not found in '{obsname}'")

        group_info = obs_group[frequency]
        time_index = pd.to_datetime(time)
        years = sorted(set(str(y) for y in time_index.year))
        
        paths = []
        for y in years:
            fname = group_info["template"].replace("%(year)", y).replace("%(var)", var)
            paths.append(os.path.join(group_info["path"], fname))

        refds = xr.open_mfdataset(paths, decode_times=True, combine='by_coords')
        if refds.lon.min() >= 0:
            refds = refds.assign_coords(lon=((refds.lon + 180) % 360 - 180)).sortby("lon")

        if var not in refds:
            raise KeyError(f"Variable '{var}' not found in file(s): {paths}")

        data = self.convert_obs_units(var, refds[var])
        data = self._ensure_datetime64(data)

        (lat1, lat2), (lon1, lon2) = self.define_region(regnam)
        data = data.sel(lat=slice(lat1, lat2), lon=slice(lon1, lon2))

        # Nearest time match within 1-day tolerance
        data = data.sel(time=time, method='nearest', tolerance=np.timedelta64(1, 'D'))

        if regional_mean:
            weights = np.cos(np.deg2rad(data.lat))
            weights /= weights.mean()
            data = (data * weights).mean(dim=["lat", "lon"])

        return data.chunk({"time": -1}) if 'time' in data.dims else data

    def convert_obs_units(self, var, data_array):
        conversions = {
            'PRECT': lambda x: x * 86400.0 * 1000.0,
            'TREFHT': lambda x: x - 273.15,
            'TS': lambda x: x - 273.15,
            'T850': lambda x: x - 273.15,
            'PSL': lambda x: x / 100.0,
            'PS': lambda x: x / 100.0,
        }

        if 'time' not in data_array.dims:
            raise ValueError(f"Expected 'time' dimension in data for '{var}'")

        return conversions[var](data_array) if var in conversions else data_array


In [5]:
class ModelDataReader:
    def __init__(self, base_path, exp_base, regnam, frequency, component="atm"):
        self.base_path = base_path
        self.exp_base = exp_base
        self.regnam = regnam
        self.frequency = frequency
        self.component = component
        self.region = self.define_region(regnam)
        self.exp_dict = self.extract_exp_info(exp_base, base_path)

    def _ensure_datetime64(self, da: xr.DataArray) -> xr.DataArray:
        """Convert cftime.datetime to datetime64[ns] if needed."""
        if not isinstance(da.indexes['time'], pd.DatetimeIndex):
            da['time'] = da.indexes['time'].to_datetimeindex(time_unit='ns')
        return da

    @staticmethod
    def define_region(regnam: str = 'global') -> Tuple[Tuple[float, float], Tuple[float, float]]:
        reg_dict = {
            'global':     [(-90, 90), (-180, 180)],
            'NHMidLat':   [(25, 50), (-60, 150)],
            'Tropics':    [(-15, 15), (-180, 180)],
            'Atlantic':   [(5, 55), (-95, -40)],
            'CONUS':      [(25, 50), (-125, -95)],
            'Antarctic':  [(-90, -50), (-180, 180)],
            'PolarN':     [(50, 90), (-180, 180)],
            'Greenland':  [(60, 85), (-75, -10)],
        }
        if regnam not in reg_dict:
            raise ValueError(f"Region '{regnam}' not defined. Available: {list(reg_dict.keys())}")
        return reg_dict[regnam]

    @staticmethod
    def extract_exp_info(base_name, data_path) -> dict:
        exps = {
            'CTRLEN10':       {'name': 'ctrl_en10', 'nens': 10, 'period': '201112'},
            'DARTEN10':       {'name': 'dart_en10', 'nens': 10, 'period': '201112'},
            'DARTEN20':       {'name': 'dart_en20', 'nens': 20, 'period': '201112'},
            'DARTEN40':       {'name': 'dart_en40', 'nens': 40, 'period': '201112'},
            'CAPTEN10_15day': {'name': 'capt_en10', 'nens': 10, 'period': '201201-201202'},
            'DARTEN20_15day': {'name': 'dart_en20', 'nens': 20, 'period': '201201-201202'},
            'DARTEN40_15day': {'name': 'dart_en40', 'nens': 40, 'period': '201201-201202'},
        }
        return {
            exp_name: {
                'run': f"{exp_name}_{base_name}" if base_name else exp_name,
                'key': exp_data['name'],
                'nens': exp_data['nens'],
                'atm': 'archive/post/atm/180x360_aave',
                'lnd': 'archive/post/lnd/180x360_aave',
                'period': exp_data['period']
            }
            for exp_name, exp_data in sorted(exps.items())
        }

    def _convert_model_units(self, var, data):
        if var == 'PRECT':
            return data * 86400.0 * 1000.0  # m/s to mm/day
        elif var in ['TREFHT', 'TS', 'T850']:
            return data - 273.15  # K to C
        elif var in ['PSL', 'PS']:
            return data / 100.0  # Pa to hPa
        return data

    def _read_variable_single_experiment(self, var: str, model_name: str, **kwargs) -> xr.DataArray:
        return self.read_variable(var, model_name, **kwargs)

    def read_variable(self, var, exp, frequency=None, regional_mean=False):
        """
        Load and process a model variable, optionally computing regional mean.
        """
        run = self.exp_dict[exp]['run']
        component_path = self.exp_dict[exp][self.component]
        nens = self.exp_dict[exp]['nens']
        period_raw = self.exp_dict[exp]['period']
        freq = frequency or self.frequency

        if freq in ['6hourly']:
            prefix = 'ts/6hourly'
            period = period_raw.split("-")[0][:4]
        elif freq in ['daily']:      
            prefix = 'ts/daily'
            period = period_raw.split("-")[0][:4]
        elif freq == 'monthly':
            prefix = 'clim/monthly'
            period = period_raw.split("-")[0][:4] + "*"
        else:
            raise ValueError(f"Unsupported frequency: {freq}")

        (lat1, lat2), (lon1, lon2) = self.region
        ensemble_data = []

        for i in range(1, nens + 1):
            member = f"EN{i:02d}"
            search_dir = os.path.join(self.base_path, run, component_path, prefix)
            file_pattern = f"{var}.{member}.{period}.nc"
            search_path = os.path.join(search_dir, file_pattern)
            rpath = sorted(glob.glob(search_path))

            if not rpath:
                raise FileNotFoundError(f"No files for {member}: {search_path}")

            ds = xcd.open_mfdataset(
                rpath,
                combine='nested',
                concat_dim='time',
                coords='minimal',
                compat='override'
            )
            
            if ds.lon.min() >= 0:
                ds = ds.assign_coords(lon=((ds.lon + 180) % 360 - 180)).sortby("lon")
                
            if var not in ds:
                raise KeyError(f"Variable '{var}' missing in {rpath[0]}")

            data = self._convert_model_units(var, ds[var])
            data = data.sel(lat=slice(lat1, lat2), lon=slice(lon1, lon2))
            
            if regional_mean:
                weights = np.cos(np.deg2rad(data.lat))
                weights /= weights.mean()
                data = (data * weights).mean(dim=["lat", "lon"])

            data = self._ensure_datetime64(data)
            ensemble_data.append(data.expand_dims(member=[i]))

        combined = xr.concat(ensemble_data, dim="member")
        
        if regional_mean:
            return combined.chunk({"member": -1, "time": -1})
        else:
            return combined.chunk({"member": -1, "time": -1, "lat": -1, "lon": -1})

    def read_variable_across_experiments(self, var: str, model_list: list, **kwargs) -> Dict[str, xr.DataArray]:
        data_dict = {}
        for model_name in model_list:
            da = self._read_variable_single_experiment(var, model_name, **kwargs)
            data_dict[model_name] = da
        return data_dict


In [6]:
class EnsembleMJOAnalyzer:
    def __init__(self, mjoindx_dir, olr_data, time, frequency, output_dir,
                 region_bounds=(-15, 15), months=None, years=None,
                 reload_mjo=False, overwrite_eofs=False):
        self.olr = olr_data
        self.time = time
        self.output_dir = output_dir
        self.region_bounds = region_bounds
        self.mjoindx_dir = mjoindx_dir
        self.overwrite_eofs = overwrite_eofs
        self.model_olr = olr_data
        self.filtered_olr = {}
        self.time_coord = "time"
        self.dt_hours = 6 if frequency == "6hourly" else 24
        self.obs_rmm = self._load_rmm(mjoindx_dir, months=months, years=years, reload_data=reload_mjo)

    @staticmethod
    def _load_rmm(mjoindx_dir, months=None, years=None, reload_data=False):
        import os
        import datetime
        local_data_txt = os.path.join(mjoindx_dir, 'rmm.74toRealtime.txt')
        local_data_nc = os.path.join(mjoindx_dir, 'rmm.74toRealtime.nc')
        col_names = ['year', 'month', 'day', 'RMM1', 'RMM2', 'phase', 'amplitude', 'Missing Value']

        if os.path.exists(local_data_nc) and not reload_data:
            ds = xr.open_dataset(local_data_nc)
        else:
            if os.path.exists(local_data_txt) and not reload_data:
                df = pd.read_csv(local_data_txt, skiprows=2, names=col_names, sep=r'\s+')
            else:
                url = 'http://www.bom.gov.au/climate/mjo/graphics/rmm.74toRealtime.txt'
                df = pd.read_csv(url, skiprows=2, names=col_names, sep=r'\s+')
                os.makedirs(os.path.dirname(local_data_txt), exist_ok=True)
                df.to_csv(local_data_txt, sep=' ', index=False, header=False)

            df.index = [datetime.datetime(y, m, d, 12) for y, m, d in zip(df.year, df.month, df.day)]
            df = df[['RMM1', 'RMM2', 'phase', 'amplitude']]
            df[df >= 999] = np.nan
            ds = xr.Dataset.from_dataframe(df).rename({'index': 'time'})
            ds.to_netcdf(local_data_nc)
        return ds

    def _bandpass_filter(self, data, low=1/100, high=1/20, order=4):
        fs = 24 / self.dt_hours
        nyq = 0.5 * fs
        b, a = butter(order, [low / nyq, high / nyq], btype='band')

        return xr.apply_ufunc(
            lambda x: filtfilt(b, a, x, axis=0),
            data,
            input_core_dims=[[self.time_coord]],
            output_core_dims=[[self.time_coord]],
            vectorize=True,
            dask="parallelized",
            output_dtypes=[data.dtype]
        )

    def filter_model_olr(self):
        for member in self.model_olr.member.values:
            da = self.model_olr.sel(member=member)
            eq_avg = da.sel(lat=slice(*self.region_bounds)).mean("lat")
            self.filtered_olr[int(member)] = self._bandpass_filter(eq_avg)

    def compute_acc_against_obs_rmm(self, start_date=None, end_date=None):
        acc_scores = {}
        rmm = self.obs_rmm.sel(time=slice(start_date, end_date)) if (start_date and end_date) else self.obs_rmm

        for m, olr in self.filtered_olr.items():
            olr_interp = olr.interp(time=rmm.time, method='nearest')
            acc1 = np.corrcoef(olr_interp.mean("lon"), rmm["RMM1"])[0, 1]
            acc2 = np.corrcoef(olr_interp.mean("lon"), rmm["RMM2"])[0, 1]
            acc_scores[m] = {"ACC_RMM1": acc1, "ACC_RMM2": acc2}
        return acc_scores

    def composite_by_rmm_phase(self, var_data, rmm_amp_thresh=1.0):
        valid = self.obs_rmm.where(self.obs_rmm.amplitude >= rmm_amp_thresh, drop=True)
        composites = {}
        for phase in range(1, 9):
            phase_times = valid.time.where(valid.phase == phase, drop=True)
            composite = var_data.sel(time=phase_times, method='nearest').mean("time")
            composites[phase] = composite
        return composites


In [7]:
if __name__ == "__main__":
    # === Setup ===
    top_path = "/compyfs/zhan391/v3_dart_cda_scratch"
    out_path = "./data"
    fig_path = "."
    mjoindx_dir = "/compyfs/zhan391/acme_init/Observations/MJO_INDEX"
    compset = "F20TR"
    resolution = "ne30pg2_r05_IcoswISC30E3r5"
    machine = "compy"
    exp_base = f"{compset}_{resolution}_{machine}"

    frequency = "daily"
    region_name = "Tropics"
    model_list = ['CAPTEN10_15day', 'DARTEN20_15day', 'DARTEN40_15day']

    for exp in model_list:
        print(f"\n[INFO] Running MJO analysis for {exp}")
        output_dir = os.path.join(out_path, f"mjo_{exp}")
        os.makedirs(output_dir, exist_ok=True)

        # === Load data using ModelDataReader ===
        reader = ModelDataReader(
            base_path=top_path,
            exp_base=exp_base,
            regnam=region_name,
            frequency=frequency,
            component="atm"
        )

        olr = reader.read_variable("FLUT", exp)  # OLR substitute
        u850 = reader.read_variable("U850", exp)
        v850 = reader.read_variable("V850", exp)
        u200 = reader.read_variable("U200", exp)
        v200 = reader.read_variable("V200", exp)
        time = olr.time.to_pandas()
        
        #print(olr.min().values,olr.max().values)
        #print(u850.min().values,u850.max().values)
        #print(v850.min().values,v850.max().values)
        #print(u200.min().values,u200.max().values)
        #print(v200.min().values,v200.max().values)

        #print(xxxx)
        
        # === Init analyzer ===
        analyzer = MJODiagnosticAnalyzer(
            mjoindx_dir=mjoindx_dir,
            olr_data=olr,
            u850_data=u850,
            v850_data=v850,
            u200_data=u200,
            v200_data=v200,
            time=time,
            frequency=frequency, 
            output_dir=output_dir,
            region_bounds=(-15, 15),
            years=[2012],
            reload_mjo=False
        )

        # === Analyze members ===
        results = analyzer.analyze_all_members()
        print(f"[INFO] Saved {len(results)} member MJO diagnostics for {exp}")



[INFO] Running MJO analysis for CAPTEN10_15day


NameError: name 'MJODiagnosticAnalyzer' is not defined